# Building GPT from First Principles - Little Shakespeare Training

**Complete implementation of GPT-style Transformer with detailed explanations**

This notebook demonstrates:
- Character-level tokenization
- Complete GPT architecture (all components)
- Training loop with proper optimization
- Text generation

**Structure**: Each section has explanations in markdown cells above the code cells.


## 1. Imports and Setup

**What we need:**
- `torch`: PyTorch for deep learning
- `torch.nn`: Neural network modules
- `torch.nn.functional`: Functional operations (softmax, etc.)
- `Dataset, DataLoader`: For data handling
- `math`: Mathematical functions (sqrt, exp, etc.)
- `requests`: To download dataset
- `Path`: File path handling


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import requests
from pathlib import Path


## 2. Character-Level Tokenizer

**Why Character-Level?**
- Task requirement: vocab_size ~65 (as per mentor's specification)
- Each character maps to unique integer ID
- Simple, no out-of-vocabulary (OOV) issues
- Model learns spelling, formatting, punctuation

**How it works:**
1. Find all unique characters in text
2. Create mappings: character → ID (encode) and ID → character (decode)
3. Vocabulary size = number of unique characters (~65 for English text)


In [ ]:
class CharTokenizer:
    """Character-level tokenizer for Shakespeare dataset (vocab_size ~65)"""
    
    def __init__(self, text):
        # Get all unique characters and sort them
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        
        # Create mappings: string to int (for encoding)
        self.stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
        
        # Create mappings: int to string (for decoding)
        self.itos = {i: ch for i, ch in enumerate(chars)}  # int to string
        
    def encode(self, text):
        """Convert string to list of integers"""
        return [self.stoi.get(c, 0) for c in text]
    
    def decode(self, ids):
        """Convert list of integers to string"""
        return ''.join([self.itos[i] for i in ids])


## 3. Layer Normalization

**Mathematical Formulation:**
```
For input x ∈ ℝ^(batch × seq_len × d):
μ = (1/d) Σᵢ xᵢ           # Mean per feature dimension
σ² = (1/d) Σᵢ (xᵢ - μ)²  # Variance per feature dimension
x̂ = (x - μ) / √(σ² + ε)  # Normalized (ε prevents division by zero)
y = γ · x̂ + β            # Scale (γ) and shift (β) are learnable
```

**Why LayerNorm?**
- Stabilizes training by normalizing activations
- Reduces sensitivity to initialization
- Works better than BatchNorm for variable-length sequences
- Enables deeper networks (pre-norm architecture)

**Key Components:**
- `eps = 1e-5`: Small constant to prevent division by zero
- `scale (γ)`: Learnable parameter to scale normalized values
- `shift (β)`: Learnable parameter to shift normalized values


In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization: Normalize across feature dimension (per sample)"""
    
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # Small constant to prevent division by zero
        
        # Learnable parameters: γ (scale) and β (shift)
        self.scale = nn.Parameter(torch.ones(emb_dim))      # γ
        self.shift = nn.Parameter(torch.zeros(emb_dim))     # β
    
    def forward(self, x):
        """Normalize input: x → (x - μ) / σ → γ·x̂ + β"""
        # Compute mean and variance across last dimension (features)
        mean = x.mean(dim=-1, keepdim=True)      # μ
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # σ²
        
        # Normalize: x̂ = (x - μ) / √(σ² + ε)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        
        # Scale and shift: y = γ·x̂ + β
        return self.scale * norm_x + self.shift


## 4. Causal Self-Attention

**Mathematical Formulation:**
```
1. Q = X · W_Q, K = X · W_K, V = X · W_V
2. S = QK^T / √d_k  (scaled dot-product attention scores)
3. M[i,j] = 0 if j≤i, -∞ if j>i  (causal mask)
4. A = softmax(S + M)  (attention weights, future positions = 0)
5. O = A · V  (weighted sum of values)
```

**Dimensions:**
- Input: x ∈ ℝ^(batch × seq_len × emb_dim)
- Output: context_vector ∈ ℝ^(batch × seq_len × out_dim)

**Key Insight:**
Each token attends only to previous tokens, enforcing autoregressive property:
- P(w_t | w_1, ..., w_{t-1}) instead of P(w_t | w_1, ..., w_n)
- This is essential for generation: during inference, future tokens don't exist!

**Why Scale by √d_k?**
- As embedding dimension grows, dot products become large
- Without scaling, softmax becomes very sharp (all probability on one token)
- Scaling prevents saturation and helps gradients flow better


In [ ]:
class CausalSelfAttention(nn.Module):
    """Causal self-attention with masking to prevent looking at future tokens"""
    
    def __init__(self, emb_dim, out_dim, context_length, drop_rate):
        super().__init__()
        # Linear projections: X → Q, K, V
        # W_Q, W_K, W_V ∈ ℝ^(emb_dim × out_dim)
        self.kw = nn.Linear(emb_dim, out_dim, bias=False)
        self.qw = nn.Linear(emb_dim, out_dim, bias=False)
        self.vw = nn.Linear(emb_dim, out_dim, bias=False)
        self.dropout = nn.Dropout(drop_rate)
        
        # Causal mask: M[i,j] = 1 if j > i (upper triangular)
        # After softmax with -∞: A[i,j] = 0 for j > i
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
    
    def forward(self, x):
        """Forward pass of causal self-attention"""
        b, t, emb_dim = x.shape
        
        # Step 1: Q = XW_Q, K = XW_K, V = XW_V
        key = self.kw(x)       # K ∈ ℝ^(b × t × out_dim)
        query = self.qw(x)     # Q ∈ ℝ^(b × t × out_dim)
        value = self.vw(x)     # V ∈ ℝ^(b × t × out_dim)
        
        # Step 2: S = QK^T (attention scores)
        # S[i,j] = similarity between token i (query) and token j (key)
        attn_score = query @ key.transpose(1, 2)  # ℝ^(b × t × t)
        
        # Step 3: Apply causal mask M: set future positions to -∞
        # After softmax: exp(-∞) = 0, so future positions have 0 attention weight
        attn_score.masked_fill_(self.mask.bool()[:t, :t], -torch.inf)
        
        # Step 4: Scale and softmax
        # Scale by √d_k prevents dot products from becoming too large
        # Softmax converts scores to probabilities: A[i,j] = P(j attends to i)
        attn_weight = torch.softmax(attn_score / math.sqrt(key.shape[-1]), dim=-1)
        attn_weight = self.dropout(attn_weight)
        
        # Step 5: Weighted sum O = AV
        # O[i] = Σ_j A[i,j] · V[j] (each output is weighted combination of values)
        context_vector = attn_weight @ value
        
        return context_vector


## 5. Multi-Head Attention

**Mathematical Formulation:**
```
Split d_model into h heads: d_k = d_model / h

For each head i:
  head_i = Attention(X · W_Q^i, X · W_K^i, X · W_V^i)

Concatenate: MultiHead = Concat(head₁, ..., head_h) · W_O
```

**Key Insight:**
Each head learns different attention patterns:
- **Head 1**: Previous token relationships (local dependencies)
- **Head 2**: Syntactic structures (grammar)
- **Head 3**: Semantic relationships (meanings)
- **Head 4**: Long-range dependencies (coreference)

**Benefits:**
- Richer representation than single head
- Same parameter count, multiple perspectives
- Parallel processing (no extra cost)
- Like having multiple experts, each with a specialty


In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention: Parallel attention heads for richer representation"""
    
    def __init__(self, emb_dim, out_dim, context_length, drop_rate, n_head):
        super().__init__()
        assert out_dim % n_head == 0, "out_dim must be divisible by n_head"
        
        # Split embedding dimension across heads
        head_dim = out_dim // n_head  # d_k = d_model / h
        
        # Create h independent attention heads
        # Each head processes same input but with different learned weights
        self.heads = nn.ModuleList([
            CausalSelfAttention(emb_dim, head_dim, context_length, drop_rate) 
            for _ in range(n_head)
        ])
        
    def forward(self, x):
        """Process input through all heads in parallel, then concatenate"""
        # Each head processes independently and learns different patterns
        head_outputs = [head(x) for head in self.heads]
        
        # Concatenate all head outputs along feature dimension
        # Result: ℝ^(batch × seq_len × out_dim)
        return torch.cat(head_outputs, dim=-1)


## 6. GELU Activation Function

**Mathematical Definition:**
```
GELU(x) = 0.5 · x · [1 + tanh(√(2/π) · (x + 0.044715x³))]
```

**Why GELU instead of ReLU?**
- **Smooth**: Differentiable everywhere (unlike ReLU)
- **Non-monotonic**: Can have negative outputs (useful for some tasks)
- **Better for NLP**: Used in GPT, BERT, and modern transformers
- **Approximation**: This is an approximation of the true GELU (which uses Gaussian CDF)


In [ ]:
class GELU(nn.Module):
    """Gaussian Error Linear Unit activation"""
    
    def forward(self, x):
        sqrt_2_over_pi = math.sqrt(2.0 / math.pi)
        return 0.5 * x * (1 + torch.tanh(sqrt_2_over_pi * (x + 0.044715 * x**3)))


## 7. Feed-Forward Network (FFN)

**Architecture:**
- Two linear layers with GELU activation
- Expansion: `emb_dim → 4×emb_dim → emb_dim`
- Applied identically to each token position

**Purpose:**
- Non-linear transformation after attention
- Allows model to learn complex patterns
- Standard in transformer architectures

**Why 4× expansion?**
- Common practice in transformers
- Provides model capacity without too many parameters
- Empirically shown to work well


In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network: two linear layers with GELU activation"""
    
    def __init__(self, emb_dim):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),  # Expand
            GELU(),                            # Activation
            nn.Linear(4 * emb_dim, emb_dim)   # Contract
        )
    
    def forward(self, x):
        return self.layer(x)


## 8. Transformer Block

**Architecture (Pre-Norm):**
```
x₁ = x₀ + Attention(LayerNorm(x₀))
x₂ = x₁ + FFN(LayerNorm(x₁))
```

**Components:**
1. **LayerNorm** → **Multi-Head Attention** → **Dropout** → **Residual**
2. **LayerNorm** → **Feed-Forward** → **Dropout** → **Residual**

**Why Residual Connections?**
- Enable gradient flow through deep networks
- Allow model to learn identity mapping (if needed)
- Prevent vanishing gradients
- Essential for training deep transformers

**Why Pre-Norm (LayerNorm before attention/FFN)?**
- More stable gradients than post-norm
- Easier to train deep networks
- Modern standard (GPT-2, GPT-3 use pre-norm)


class TransformerBlock(nn.Module):
    """Transformer block: Multi-head attention + Feed-forward with residual connections"""
    
    def __init__(self, cfg):
        super().__init__()
        self.layer_norm1 = LayerNorm(cfg['emb_dim'])
        self.mha = MultiHeadAttention(
            cfg['emb_dim'], cfg['emb_dim'], 
            cfg['context_length'], cfg['drop_rate'], cfg['n_head']
        )
        self.dropout = nn.Dropout(cfg['drop_rate'])
        self.layer_norm2 = LayerNorm(cfg['emb_dim'])
        self.ff = FeedForward(cfg['emb_dim'])
    
    def forward(self, x):
        """Pre-norm architecture with residual connections"""
        # First sub-layer: Attention
        shortcut = x
        x = self.layer_norm1(x)      # Pre-norm
        x = self.mha(x)              # Multi-head attention
        x = self.dropout(x)          # Regularization
        x = x + shortcut             # Residual connection
        
        # Second sub-layer: Feed-forward
        shortcut = x
        x = self.layer_norm2(x)      # Pre-norm
        x = self.ff(x)              # Feed-forward network
        x = self.dropout(x)          # Regularization
        x = x + shortcut             # Residual connection
        
        return x


## 9. Complete GPT Model

**Architecture Flow:**
```
Input Tokens (IDs) 
  ↓
Token Embeddings + Positional Embeddings
  ↓
Dropout
  ↓
[N × Transformer Blocks]
  ↓
Final LayerNorm
  ↓
Output Head (Linear layer)
  ↓
Logits [vocab_size]
```

**Key Components:**
1. **Token Embeddings**: Each token ID → dense vector
2. **Positional Embeddings**: Each position → dense vector (learnable)
3. **Transformer Blocks**: Stack of attention + FFN layers
4. **Final LayerNorm**: Stabilize output before prediction
5. **Output Head**: Linear layer mapping to vocabulary size

**Why Add Positional Embeddings?**
- Attention is permutation-invariant (order doesn't matter)
- Positional embeddings inject order information
- Critical for language (word order matters!)


In [ ]:
class GPTModel(nn.Module):
    """Complete GPT model: embeddings + transformer blocks + output head"""
    
    def __init__(self, cfg):
        super().__init__()
        # Embeddings
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        
        # Transformer blocks: Stack N identical blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])]
        )
        
        # Final layer norm and output head
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.output_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'])
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        
        # Token and positional embeddings
        tok_emb = self.tok_emb(in_idx)  # [batch × seq_len × emb_dim]
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # [seq_len × emb_dim]
        
        # Broadcast positional embeddings to batch dimension
        x = tok_emb + pos_emb.unsqueeze(0)  # [batch × seq_len × emb_dim]
        x = self.drop_emb(x)
        
        # Pass through transformer blocks
        x = self.trf_blocks(x)  # [batch × seq_len × emb_dim]
        
        # Final layer norm and output logits
        x = self.final_norm(x)
        logits = self.output_head(x)  # [batch × seq_len × vocab_size]
        
        return logits


## DataLoader SetUp

In [ ]:
        
class ShakespeareDataset(Dataset):
    """Dataset for Little Shakespeare with character-level tokenization"""
    
    def __init__(self, text, tokenizer, block_size, stride):
        self.block_size = block_size
        self.tokenizer = tokenizer
        
        # Tokenize entire text into integer IDs
        token_ids = tokenizer.encode(text)
        
        # Create input-target pairs using sliding window
        self.input_ids = []
        self.target_ids = []
        
        # Sliding window: for each position, create input-target pair
        for i in range(0, len(token_ids) - block_size, stride):
            input_chunk = token_ids[i:i + block_size]      # Input: tokens[i:i+block_size]
            target_chunk = token_ids[i + 1:i + block_size + 1]  # Target: shifted by 1
            
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]



## 11. Download Shakespeare Dataset

**Purpose:**
- Download Little Shakespeare dataset (~1 MB)
- Contains works of Shakespeare in plain text

**Why Little Shakespeare?**
- Rich in grammar, dialogue, poetry
- Good for language modeling
- Character-level tokenization makes it educational
- Small enough to train quickly

**Dataset Details:**
- Source: Karpathy's char-rnn repository
- Size: ~1 MB of text
- Contains full works of Shakespeare


In [ ]:
def download_shakespeare(data_path):
    """Download Little Shakespeare dataset"""
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    
    if not data_path.exists():
        data_path.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading Little Shakespeare dataset...")
        response = requests.get(url)
        data_path.write_text(response.text)
        print(f"Downloaded {len(response.text)} characters")
    else:
        print(f"Dataset already exists: {len(data_path.read_text())} characters")

# Download dataset
data_path = Path("data/tinyshakespeare.txt")
download_shakespeare(data_path)


## 12. Load Data and Create Tokenizer

**Steps:**
1. Load text from downloaded file
2. Create character-level tokenizer
3. Split into train/validation (90/10)
4. Verify vocabulary size (~65 characters)

**Why 90/10 Split?**
- Training: 90% of data for learning patterns
- Validation: 10% for monitoring overfitting
- Standard practice in machine learning
- Hold-out set for evaluation


In [ ]:
# Load text
text = data_path.read_text()
print(f"Dataset size: {len(text):,} characters")
print(f"First 200 characters:\n{text[:200]}")

# Create tokenizer
tokenizer = CharTokenizer(text)
print(f"\nVocabulary size: {tokenizer.vocab_size}")
print(f"Characters: {''.join(tokenizer.itos.values())}")

# Split into train and validation (90/10)
n = len(text)
train_text = text[:int(n * 0.9)]
val_text = text[int(n * 0.9):]

print(f"\nTrain size: {len(train_text):,} characters")
print(f"Val size: {len(val_text):,} characters")


## 13. Model Configuration

**GPT Configuration for Shakespeare:**
- Smaller model suitable for character-level and limited compute
- **Vocab Size**: ~65 (character-level, as per task requirement)
- **Context Length**: 128 (sequence length)
- **Embedding Dimension**: 384
- **Attention Heads**: 6
- **Transformer Layers**: 6
- **Parameters**: ~8M total

**Why This Configuration?**
- Character-level requires smaller vocab than word-level
- Smaller context length for faster training
- Enough capacity to learn patterns, but not too large
- Can run on consumer GPUs


In [ ]:
# Model configuration
GPT_CONFIG = {
    'vocab_size': tokenizer.vocab_size,  # ~65 for character-level
    'context_length': 128,  # Block size (sequence length)
    'emb_dim': 384,         # Embedding dimension
    'n_head': 6,            # Number of attention heads
    'n_layers': 6,          # Number of transformer blocks
    'drop_rate': 0.1,       # Dropout rate
    'qkv_bias': False       # Query/Key/Value bias (False as per GPT-2)
}

print("\nModel Configuration:")
for key, value in GPT_CONFIG.items():
    print(f"  {key}: {value}")
